# Phase 1: Data Preparation & Feature Validation
**Notebook:** `01_data_prep.ipynb`  
**Goal:** Ingest raw multi-source monthly tourism dataset (`data/raw/Dataset.xlsx`), normalize timestamps, validate exogenous signals and shock regimes, generate temporal lag and cyclical trigonometric features, identify ML-ready observations, and persist the canonical datasets to `data/processed/tourism_dataset_final.csv` and `backend/data/historical.csv`.

## 1. Imports & Path Configuration

In [1]:
import os
import numpy as np
import pandas as pd

# Automatically resolve paths whether executed from root or notebooks/
BASE_DIR = '..' if os.path.exists('../data/raw/Dataset.xlsx') else '.'

RAW_XLSX = os.path.join(BASE_DIR, 'data', 'raw', 'Dataset.xlsx')
OUT_CSV  = os.path.join(BASE_DIR, 'data', 'processed', 'tourism_dataset_final.csv')
HIST_CSV = os.path.join(BASE_DIR, 'backend', 'data', 'historical.csv')

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
os.makedirs(os.path.dirname(HIST_CSV), exist_ok=True)

print(f"Raw dataset path: {RAW_XLSX}")
print(f"Processed CSV target: {OUT_CSV}")
print(f"Backend CSV target: {HIST_CSV}")

Raw dataset path: ../data/raw/Dataset.xlsx
Processed CSV target: ../data/processed/tourism_dataset_final.csv
Backend CSV target: ../backend/data/historical.csv


## 2. Step 1.1: Load Master Dataset (Raw)

In [2]:
df_raw = pd.read_excel(RAW_XLSX, sheet_name='Dataset')
print(f"Raw Dataset Shape: {df_raw.shape}")
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
df_raw.head()

Raw Dataset Shape: (132, 17)
Columns (17): ['date', 'year', 'month', 'tourist_arrivals', 'usd_lkr_rate_monthly', 'gtrends_flights', 'gtrends_hotels', 'gtrends_visa', 'shock_flag', 'shock_type', 'month_sin', 'month_cos', 'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12', 'arrivals_rolling_mean_3', 'dataset_split']


,date,year,month,tourist_arrivals,usd_lkr_rate_monthly,gtrends_flights,gtrends_hotels,gtrends_visa,shock_flag,shock_type,month_sin,month_cos,arrivals_lag_1,arrivals_lag_3,arrivals_lag_12,arrivals_rolling_mean_3,dataset_split
0,2015-01-01,2015,1,156246,131.55,16,27,37,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
1,2015-02-01,2015,2,165541,132.73,13,25,38,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
2,2015-03-01,2015,3,157051,132.90,12,22,34,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
3,2015-04-01,2015,4,122217,132.90,12,24,31,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train
4,2015-05-01,2015,5,113529,133.50,12,22,31,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train


## 3. Step 1.2: Fix and Normalize Date Column
Excel dates can be stored as integer serial offsets (days from 1899-12-30) or strings.
We normalize all dates to standard datetime aligned at the 1st of each month.

In [3]:
df = df_raw.copy()

if pd.api.types.is_numeric_dtype(df['date']):
    df['date'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(df['date'].astype(int), unit='D')
else:
    df['date'] = pd.to_datetime(df['date'])

# Normalize to month start (1st day of month)
df['date'] = df['date'].dt.to_period('M').dt.to_timestamp()
df = df.sort_values('date').reset_index(drop=True)

print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
print(f"Total Monthly Timepoints: {len(df)}")

Date Range: 2015-01-01 to 2025-12-01
Total Monthly Timepoints: 132


## 4. Step 1.3: Data Type Conversions & Missingness Audit

In [4]:
# Clean shock_type string placeholders
df['shock_type'] = df['shock_type'].replace(['None', 'none', 'NaN', ''], np.nan)

# Ensure numeric columns are cast accurately
raw_numeric_cols = [
    'year', 'month', 'tourist_arrivals', 'usd_lkr_rate_monthly',
    'gtrends_flights', 'gtrends_hotels', 'gtrends_visa', 'shock_flag'
]
for col in raw_numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("=== Missing Value Count per Column ===")
print(df[raw_numeric_cols + ['shock_type', 'dataset_split']].isnull().sum())

=== Missing Value Count per Column ===
year                     0
month                    0
tourist_arrivals         0
usd_lkr_rate_monthly     0
gtrends_flights          0
gtrends_hotels           0
gtrends_visa             0
shock_flag               0
shock_type              97
dataset_split            0
dtype: int64


## 5. Step 1.4: Feature Engineering (Cyclical Encoding & Lags)
We construct:
- **Cyclical Month Encoding**: $\sin(2\pi \times \text{month} / 12)$ and $\cos(2\pi \times \text{month} / 12)$
- **Autoregressive Lags**: $t-1, t-3, t-12$
- **Rolling Window Statistics**: 3-month trailing mean

In [5]:
# 1. Cyclical seasonality
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

# 2. Lag features
df['arrivals_lag_1'] = df['tourist_arrivals'].shift(1)
df['arrivals_lag_3'] = df['tourist_arrivals'].shift(3)
df['arrivals_lag_12'] = df['tourist_arrivals'].shift(12)

# 3. Rolling mean
df['arrivals_rolling_mean_3'] = df['tourist_arrivals'].rolling(window=3).mean()

print("Feature Engineering complete.")
df[['date', 'tourist_arrivals', 'month_sin', 'month_cos', 'arrivals_lag_1', 'arrivals_lag_12', 'arrivals_rolling_mean_3']].head(15)

Feature Engineering complete.


,date,tourist_arrivals,month_sin,month_cos,arrivals_lag_1,arrivals_lag_12,arrivals_rolling_mean_3
0,2015-01-01,156246,5.000000e-01,8.660254e-01,NaN,NaN,NaN
1,2015-02-01,165541,8.660254e-01,5.000000e-01,156246.0,NaN,NaN
2,2015-03-01,157051,1.000000e+00,6.123234e-17,165541.0,NaN,159612.666667
3,2015-04-01,122217,8.660254e-01,-5.000000e-01,157051.0,NaN,148269.666667
4,2015-05-01,113529,5.000000e-01,-8.660254e-01,122217.0,NaN,130932.333333
5,2015-06-01,115467,1.224647e-16,-1.000000e+00,113529.0,NaN,117071.000000
6,2015-07-01,175804,-5.000000e-01,-8.660254e-01,115467.0,NaN,134933.333333
7,2015-08-01,166610,-8.660254e-01,-5.000000e-01,175804.0,NaN,152627.000000
8,2015-09-01,143374,-1.000000e+00,-1.836970e-16,166610.0,NaN,161929.333333
9,2015-10-01,132280,-8.660254e-01,5.000000e-01,143374.0,NaN,147421.333333


## 6. Step 1.5: Validate Shock Flags & Train/Test Splits

In [6]:
print("--- Shock Flags Breakdown ---")
print(df['shock_flag'].value_counts())
print("\n--- Shock Types Breakdown ---")
print(df['shock_type'].value_counts(dropna=False))
print("\n--- Train / Test Split Breakdown ---")
print(df['dataset_split'].value_counts())

--- Shock Flags Breakdown ---
shock_flag
0    97
1    35
Name: count, dtype: int64

--- Shock Types Breakdown ---
shock_type
NaN                97
COVID-19           22
Economic Crisis    12
Easter Attack       1
Name: count, dtype: int64

--- Train / Test Split Breakdown ---
dataset_split
train    84
test     48
Name: count, dtype: int64


## 7. Step 1.6: Mark ML-Ready Observations
Because `arrivals_lag_12` requires 12 months of prior history, the first 12 months (2015) have NaN in lag-12.
We flag rows with complete feature vectors as `ml_ready = 1`.

In [7]:
ml_feature_cols = [
    'usd_lkr_rate_monthly', 'gtrends_flights', 'gtrends_hotels', 'gtrends_visa',
    'shock_flag', 'month_sin', 'month_cos',
    'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12', 'arrivals_rolling_mean_3'
]

df['ml_ready'] = df[ml_feature_cols].notna().all(axis=1).astype(int)

print(f"Total observations: {len(df)}")
print(f"ML-ready observations (2016-2025): {df['ml_ready'].sum()}")
print(f"Non ML-ready warmup observations (2015): {(df['ml_ready'] == 0).sum()}")

Total observations: 132
ML-ready observations (2016-2025): 120
Non ML-ready warmup observations (2015): 12


## 8. Step 1.7: Save Canonical Processed Dataset

In [8]:
final_column_order = [
    'date', 'year', 'month', 'tourist_arrivals',
    'usd_lkr_rate_monthly',
    'gtrends_flights', 'gtrends_hotels', 'gtrends_visa',
    'shock_flag', 'shock_type',
    'month_sin', 'month_cos',
    'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12',
    'arrivals_rolling_mean_3',
    'dataset_split', 'ml_ready'
]

df_final = df[final_column_order].copy()

# Save to processed data directory and backend data directory
df_final.to_csv(OUT_CSV, index=False)
df_final.to_csv(HIST_CSV, index=False)

print(f"Successfully saved canonical dataset: {OUT_CSV} ({df_final.shape})")
print(f"Successfully saved backend API copy: {HIST_CSV} ({df_final.shape})")

Successfully saved canonical dataset: ../data/processed/tourism_dataset_final.csv ((132, 18))
Successfully saved backend API copy: ../backend/data/historical.csv ((132, 18))


## 9. Verification Summary

In [9]:
print("=== SUMMARY AUDIT ===")
print(f"Date range        : {df_final['date'].min().date()} to {df_final['date'].max().date()}")
print(f"Total observations: {len(df_final)}")
print(f"Training months   : {(df_final['dataset_split'] == 'train').sum()} (ML-ready train: {((df_final['dataset_split'] == 'train') & (df_final['ml_ready'] == 1)).sum()})")
print(f"Testing months    : {(df_final['dataset_split'] == 'test').sum()} (ML-ready test: {((df_final['dataset_split'] == 'test') & (df_final['ml_ready'] == 1)).sum()})")
print(f"Shock months      : {int(df_final['shock_flag'].sum())}")
print(f"Total ML-ready    : {int(df_final['ml_ready'].sum())}")
print("\nFirst 3 processed rows:")
print(df_final.head(3))
print("\nLast 3 processed rows:")
print(df_final.tail(3))
print("\nPhase 1 Data Preparation COMPLETE!")

=== SUMMARY AUDIT ===
Date range        : 2015-01-01 to 2025-12-01
Total observations: 132
Training months   : 84 (ML-ready train: 72)
Testing months    : 48 (ML-ready test: 48)
Shock months      : 35
Total ML-ready    : 120

First 3 processed rows:
        date  year  month  tourist_arrivals  usd_lkr_rate_monthly  \
0 2015-01-01  2015      1            156246                131.55   
1 2015-02-01  2015      2            165541                132.73   
2 2015-03-01  2015      3            157051                132.90   

   gtrends_flights  gtrends_hotels  gtrends_visa  shock_flag shock_type  \
0               16              27            37           0        NaN   
1               13              25            38           0        NaN   
2               12              22            34           0        NaN   

   month_sin     month_cos  arrivals_lag_1  arrivals_lag_3  arrivals_lag_12  \
0   0.500000  8.660254e-01             NaN             NaN              NaN   
1   0.866025  